# Benchmark de Cascata Adaptativa — scaffold do Marco 1

Validação no Google Colab (GPU NVIDIA T4) dos Componentes B, C e F do PR `003`:

- **B** — servidor vLLM com prefix caching e `--logprobs-mode raw_logprobs` (Story 1 do `engine-router-spec`)
- **C** — System 1: decisão tipada de 1 token via `/v1/systemone` (Story 2 do `engine-router-spec`)
- **F** — System 2: CoT + JSON via `/v1/chat/completions` (Story 1 do `system2-evaluator-spec`)

**Como usar**: em *Runtime → Change runtime type*, selecione **T4 GPU** e rode as células em ordem. O benchmark completo (Componente G: cenários, sweep de α, `results/`) estende este notebook na fase seguinte.

Nota de reprodutibilidade: a documentação do vLLM avisa que resultados podem variar entre requests idênticas (batching/prefix cache). `VLLM_BATCH_INVARIANT=1` **não** é configurado porque o benchmark mede latência realista.

## Setup — clone e dependências

Clona o repo público e instala `requirements-colab.txt` (inclui o vLLM; a instalação demora alguns minutos).

In [ ]:
import os
import subprocess

REPO = "adaptive-cascade-routing"
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "https://github.com/Grazinascito/adaptive-cascade-routing.git"],
        check=True,
    )

In [ ]:
%cd adaptive-cascade-routing
!git log --oneline -1
!pip install -q -r requirements-colab.txt

In [ ]:
import sys

sys.path.insert(0, os.getcwd())

from src.data.loader import format_system1_prompt, get_frozen_eval_set
from src.engine import server, system1, system2

print("imports ok — src/ nunca importa vllm; o servidor roda como subprocesso")

## Componente B — servidor vLLM (Story 1)

Sobe o servidor com `Qwen/Qwen2.5-7B-Instruct-AWQ` (AWQ 4-bit), prefix caching e 90% da VRAM (NFR-002: ≤ 14.4 GB da T4 de 16 GB). Na primeira execução o vLLM baixa ~5.5 GB de pesos, por isso o timeout de startup aqui é 1800 s. Se algo falhar, o log completo do servidor fica em `vllm_server.log`.

In [ ]:
started = server.start_server(system1.DEFAULT_MODEL, startup_timeout_s=1800.0)
print("start_server:", started)
print("health_check:", server.health_check())
print("log do servidor:", server.VLLM_LOG_PATH)

In [ ]:
warmup = format_system1_prompt("Um cachorro corre no parque.", "O cachorro está correndo.")
latencies = []
for i in range(10):
    output = system1.evaluate_typed_decision(warmup["user_state"], warmup["choices"])
    latencies.append(output["latency_ms"])
    print(f"warmup {i + 1:>2}: {output['latency_ms']:.1f} ms")

rest_avg = sum(latencies[1:]) / len(latencies[1:])
drop = 100.0 * (latencies[0] - rest_avg) / latencies[0]
print(f"\nqueda média das queries 2-10 vs a 1ª: {drop:.1f}% (Independent Test: >= 70%)")

In [ ]:
!nvidia-smi
print("limite NFR-002: 14.4 GB (90% da T4 de 16 GB) para pesos + KV cache")

## Componente C — System 1 (Story 2)

50 amostras balanceadas do frozen set pelo caminho fast-path. Alvo NFR-001: p95 < 60 ms (valor medido é informativo — depende do hardware do momento); a acurácia é reportada para contexto.

In [ ]:
import statistics

samples = get_frozen_eval_set(sample_limit=50)
print(f"{len(samples)} amostras balanceadas do frozen set")

latencies = []
correct = 0
first_output = None
for sample in samples:
    payload = format_system1_prompt(sample["premise"], sample["hypothesis"])
    output = system1.evaluate_typed_decision(payload["user_state"], payload["choices"])
    latencies.append(output["latency_ms"])
    correct += output["top_choice"] == sample["label"]
    if first_output is None:
        first_output = output

qs = statistics.quantiles(latencies, n=100)
print(f"p50={qs[49]:.1f} ms | p95={qs[94]:.1f} ms | p99={qs[98]:.1f} ms (alvo p95 < 60 ms)")
print(f"acurácia System 1: {correct}/{len(samples)} = {correct / len(samples):.1%}")
print(
    "exemplo de saída:",
    {key: first_output[key] for key in ("top_choice", "top_probability", "entropy")},
)

## Componente F — System 2 (Story 1)

20 pares pelo slow-path (CoT + JSON estrito). Independent Test: ≥ 95% de respostas com JSON válido sem precisar do regex de fallback.

In [ ]:
pairs = get_frozen_eval_set(sample_limit=20)
outputs = [system2.execute_reasoning(s["premise"], s["hypothesis"]) for s in pairs]

valid = sum(not o["fallback_used"] for o in outputs)
print(f"JSON válido sem regex: {valid}/{len(outputs)} = {valid / len(outputs):.1%} (limiar >= 95%)")
lat = [o["latency_ms"] for o in outputs]
print(f"latência System 2 — min {min(lat):.0f} ms / max {max(lat):.0f} ms")
for o in outputs[:2]:
    print("-", o["label"], "| fallback:", o["fallback_used"], "|", o["reasoning"][:120])

## Próxima fase (fora deste scaffold)

Componente G (`src/metrics/evaluator.py`): orquestra os três cenários (System 1 puro, System 2 puro, cascata com α), sweep de α ∈ {0.50, 0.70, 0.85, 0.90, 0.95} e grava `results/`. Este notebook será estendido com essas células.